In [56]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score

# 데이터 불러오기, 불필요한 첫번째, 두번째 열 제거

In [58]:
train_data = pd.read_csv("./Data/train.csv")
test_data = pd.read_csv("./Data/test.csv")
specification_data = pd.read_excel("./Data/데이터 명세.xlsx")

In [59]:
train_data = train_data.iloc[:,2:]
test_data = test_data.iloc[:,2:]

In [60]:
categorical_columns = specification_data[specification_data['범주형 여부'] == 1]['컬럼명'].tolist()
numerical_columns = specification_data[specification_data['범주형 여부'] == 0]['컬럼명'].tolist()
categorical_columns.append("특정 시술 유형")
categorical_columns

['시술 시기 코드',
 '시술 당시 나이',
 '시술 유형',
 '배란 자극 여부',
 '배란 유도 유형',
 '단일 배아 이식 여부',
 '착상 전 유전 검사 사용 여부',
 '착상 전 유전 진단 사용 여부',
 '남성 주 불임 원인',
 '남성 부 불임 원인',
 '여성 주 불임 원인',
 '여성 부 불임 원인',
 '부부 주 불임 원인',
 '부부 부 불임 원인',
 '불명확 불임 원인',
 '불임 원인 - 난관 질환',
 '불임 원인 - 남성 요인',
 '불임 원인 - 배란 장애',
 '불임 원인 - 여성 요인',
 '불임 원인 - 자궁경부 문제',
 '불임 원인 - 자궁내막증',
 '불임 원인 - 정자 농도',
 '불임 원인 - 정자 면역학적 요인',
 '불임 원인 - 정자 운동성',
 '불임 원인 - 정자 형태',
 '배아 생성 주요 이유',
 '총 시술 횟수',
 '클리닉 내 총 시술 횟수',
 'IVF 시술 횟수',
 'DI 시술 횟수',
 '총 임신 횟수',
 'IVF 임신 횟수',
 'DI 임신 횟수',
 '총 출산 횟수',
 'IVF 출산 횟수',
 'DI 출산 횟수',
 '난자 출처',
 '정자 출처',
 '난자 기증자 나이',
 '정자 기증자 나이',
 '동결 배아 사용 여부',
 '신선 배아 사용 여부',
 '기증 배아 사용 여부',
 '대리모 여부',
 'PGD 시술 여부',
 'PGS 시술 여부',
 '특정 시술 유형']

# 결측치 처리

In [62]:
# 수치형 변수 결측치 처리 (중앙값 대체)
num_imputer = SimpleImputer(strategy="median")
train_data[numerical_columns] = num_imputer.fit_transform(train_data[numerical_columns])
test_data[numerical_columns] = num_imputer.transform(test_data[numerical_columns])

In [63]:
# 범주형 변수 결측치 처리 (최빈값 대체)
categorical_columns = [col for col in categorical_columns if col in train_data.columns]
cat_imputer = SimpleImputer(strategy="most_frequent")
train_data[categorical_columns] = cat_imputer.fit_transform(train_data[categorical_columns])
test_data[categorical_columns] = cat_imputer.transform(test_data[categorical_columns])

# 인코딩

In [65]:
# 범주형 변수 Ordinal Encoding 적용
ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
train_data[categorical_columns] = ordinal_encoder.fit_transform(train_data[categorical_columns])
test_data[categorical_columns] = ordinal_encoder.transform(test_data[categorical_columns])

# 데이터 분할

In [67]:
X = train_data.drop('임신 성공 여부', axis=1)
y = train_data['임신 성공 여부']

# train-validation분리
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state= 42, stratify=y)

# 모델 학습 및 성능 평가

* 73.45%

In [70]:
model = RandomForestClassifier(random_state=42, n_estimators=100, n_jobs=1)
model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=1, random_state=42)

In [71]:
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)

print("random forest validation set accuracy : {:.2f}%".format(accuracy_score(y_val,y_pred)*100))

random forest validation set accuracy : 71.49%
